In [ ]:
import dataclasses
import math
import warnings
from typing import Callable
import os

import lovely_tensors
import numpy as np
import PIL.Image
import torch
import torch.nn.functional as F
import torchvision.transforms as TVT
import torchvision.transforms.functional as TVTF
import tqdm
from omegaconf import OmegaConf
from torch import Tensor, nn
from torchmetrics.classification import MulticlassJaccardIndex

DINOv3_REPO_DIR = r"E:\Python\pythonProjecttest\demo161\dinov3-main"
import os
os.environ["TORCH_HOME"] = r"E:\Python\pythonProjecttest\demo161"

# 准备数据集

In [52]:
# 请在两个数据集类的 __init__ 中，将 `self.ds` 改成你自己的数据集。

class ZeroShotSegmentationDataset(torch.utils.data.Dataset):
    CLASS_NAMES: tuple[str, ...]
    IGNORE_ZERO_LABEL: bool  # 若为 True，则把标签 0 映射为 255（忽略），其他标签整体减 1
    transform: Callable[[PIL.Image.Image], Tensor]

    def __init__(self, transform: Callable[[PIL.Image.Image], Tensor]) -> None:
        self.transform = transform

    def _mask_to_tensor(self, mask_pil: PIL.Image.Image) -> Tensor:
        mask = torch.from_numpy(np.array(mask_pil)).long()
        if self.IGNORE_ZERO_LABEL:
            mask = torch.where((mask == 0) | (mask == 255), 255, mask - 1)
        return mask

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        img, target = self.ds[idx]
        img = self.transform(img)
        target = self._mask_to_tensor(target)
        return img, target

    def __len__(self) -> int:
        return len(self.ds)


class Cityscapes(ZeroShotSegmentationDataset):
    CLASS_NAMES = (
        "road",
        "sidewalk",
        "building",
        "wall",
        "fence",
        "pole",
        "traffic light",
        "traffic sign",
        "vegetation",
        "terrain",
        "sky",
        "person",
        "rider",
        "car",
        "truck",
        "bus",
        "train",
        "motorcycle",
        "bicycle",
    )
    IGNORE_ZERO_LABEL = False

    def __init__(self, transform: Callable[[PIL.Image.Image], Tensor]) -> None:
        super().__init__(transform)
        self.ds = None # 在这里放 "Cityscapes:split=VAL" 数据集


class Ade20k(ZeroShotSegmentationDataset):
    CLASS_NAMES = (
        "wall",
        "building",
        "sky",
        "floor",
        "tree",
        "ceiling",
        "road",
        "bed ",
        "windowpane",
        "grass",
        "cabinet",
        "sidewalk",
        "person",
        "earth",
        "door",
        "table",
        "mountain",
        "plant",
        "curtain",
        "chair",
        "car",
        "water",
        "painting",
        "sofa",
        "shelf",
        "house",
        "sea",
        "mirror",
        "rug",
        "field",
        "armchair",
        "seat",
        "fence",
        "desk",
        "rock",
        "wardrobe",
        "lamp",
        "bathtub",
        "railing",
        "cushion",
        "base",
        "box",
        "column",
        "signboard",
        "chest of drawers",
        "counter",
        "sand",
        "sink",
        "skyscraper",
        "fireplace",
        "refrigerator",
        "grandstand",
        "path",
        "stairs",
        "runway",
        "case",
        "pool table",
        "pillow",
        "screen door",
        "stairway",
        "river",
        "bridge",
        "bookcase",
        "blind",
        "coffee table",
        "toilet",
        "flower",
        "book",
        "hill",
        "bench",
        "countertop",
        "stove",
        "palm",
        "kitchen island",
        "computer",
        "swivel chair",
        "boat",
        "bar",
        "arcade machine",
        "hovel",
        "bus",
        "towel",
        "light",
        "truck",
        "tower",
        "chandelier",
        "awning",
        "streetlight",
        "booth",
        "television receiver",
        "airplane",
        "dirt track",
        "apparel",
        "pole",
        "land",
        "bannister",
        "escalator",
        "ottoman",
        "bottle",
        "buffet",
        "poster",
        "stage",
        "van",
        "ship",
        "fountain",
        "conveyer belt",
        "canopy",
        "washer",
        "plaything",
        "swimming pool",
        "stool",
        "barrel",
        "basket",
        "waterfall",
        "tent",
        "bag",
        "minibike",
        "cradle",
        "oven",
        "ball",
        "food",
        "step",
        "tank",
        "trade name",
        "microwave",
        "pot",
        "animal",
        "bicycle",
        "lake",
        "dishwasher",
        "screen",
        "blanket",
        "sculpture",
        "hood",
        "sconce",
        "vase",
        "traffic light",
        "tray",
        "ashcan",
        "fan",
        "pier",
        "crt screen",
        "plate",
        "monitor",
        "bulletin board",
        "shower",
        "radiator",
        "glass",
        "clock",
        "flag",
    )
    IGNORE_ZERO_LABEL = True

    def __init__(self, transform: Callable[[PIL.Image.Image], Tensor]) -> None:
        super().__init__(transform)
        self.ds = None # 在这里放 "ADE20KChallengeData2016:split=VAL" 数据集


DATASETS: dict[str, type[ZeroShotSegmentationDataset]] = {
    "cityscapes": Cityscapes,
    "ade20k": Ade20k,
}
NORMALIZE_IMAGENET = TVT.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

# 图像编码函数

In [59]:
def encode_image(model, img: Tensor) -> tuple[Tensor, Tensor]:
    """从主干与附加块提取图像特征。"""
    B, _, H, W = img.shape
    P = model.visual_model.backbone.patch_size # DINOv3 的 patch 大小
    new_H = math.ceil(H / P) * P
    new_W = math.ceil(W / P) * P

    # 将图像拉伸到 patch 大小的整数倍
    if (H, W) != (new_H, new_W):
        img = F.interpolate(img, size=(new_H, new_W), mode="bicubic", align_corners=False)  # [B, 3, H', W']

    B, _, h_i, w_i = img.shape

    backbone_patches = None
    cls_tokens, _, patch_tokens = model.visual_model.get_class_and_patch_tokens(img)
    blocks_patches = (
        patch_tokens.reshape(B, h_i // P, w_i // P, -1).contiguous()
    ) # [1, h, w, D]

    return backbone_patches, blocks_patches


class ShortSideResize(nn.Module):
    def __init__(self, size: int, interpolation: TVT.InterpolationMode) -> None:
        super().__init__()
        self.size = size
        self.interpolation = interpolation

    def forward(self, img: Tensor) -> Tensor:
        _, h, w = TVTF.get_dimensions(img)
        if (w <= h and w == self.size) or (h <= w and h == self.size):
            return img
        if w < h:
            new_w = self.size
            new_h = int(self.size * h / w)
            return TVTF.resize(img, [new_h, new_w], self.interpolation)
        else:
            new_h = self.size
            new_w = int(self.size * w / h)
            return TVTF.resize(img, [new_h, new_w], self.interpolation)

# 全图或滑窗推理的函数

In [64]:
def predict_whole(model, img: Tensor, text_features: Tensor) -> Tensor:
    # 只用附加块的图像特征，忽略 backbone 特征
    _, H, W = img.shape
    _, blocks_feats = encode_image(model, img.unsqueeze(0))  # [1, h, w, D]
    _, h, w, _ = blocks_feats.shape
    blocks_feats = blocks_feats.squeeze(0)  # [h, w, D]

    # 计算 patch 特征与文本特征的余弦相似度（已归一化）
    blocks_feats = F.normalize(blocks_feats, p=2, dim=-1)  # [h, w, D]
    cos = torch.einsum("cd,hwd->chw", text_features, blocks_feats)  # [num_classes, h, w]

    # 返回低分辨率的相似度，后续会上采样到目标分辨率
    return cos

def predict_slide(model, img: Tensor, text_features: Tensor, side: int, stride: int) -> Tensor:
    # 以滑窗方式遍历重叠窗口，在原始分辨率累积预测
    _, H, W = img.shape
    num_classes, _ = text_features.shape
    probs = torch.zeros([num_classes, H, W], device="cuda")
    counts = torch.zeros([H, W], device="cuda")
    h_grids = max(H - side + stride - 1, 0) // stride + 1
    w_grids = max(W - side + stride - 1, 0) // stride + 1
    for i in range(h_grids):
        for j in range(w_grids):
            y1 = i * stride
            x1 = j * stride
            y2 = min(y1 + side, H)
            x2 = min(x1 + side, W)
            y1 = max(y2 - side, 0)
            x1 = max(x2 - side, 0)

            # 对该窗口计算余弦相似度，逻辑同 predict_whole
            img_window = img[:, y1:y2, x1:x2]  # [3, H_win, W_win]
            cos = predict_whole(model, img_window, text_features)  # [num_classes, h, w]

            # 上采样到窗口分辨率并累积“概率”
            # 注意：这不是真概率，只是对余弦相似度做 softmax 的结果
            cos = F.interpolate(
                cos.unsqueeze(0),
                size=img_window.shape[1:],
                mode="bilinear",
                align_corners=False,
            ).squeeze(0)  # [num_classes, H_win, W_win]
            probs[:, y1:y2, x1:x2] += cos.softmax(dim=0)  # [num_classes, h, w]
            counts[y1:y2, x1:x2] += 1
    probs /= counts

    # 返回图像分辨率下的“概率”，后续会上采样到目标分辨率
    return probs  # [num_classes, H, W]


# 提示模板

In [65]:
# 低显存模式配置（可按需调整）
cfg.resize = 384   # 缩短边，减轻显存
cfg.mode = "whole"  # 改全图推理，减少中间特征
cfg.side = 256
cfg.stride = 192
print(f"Override cfg: resize={cfg.resize}, mode={cfg.mode}, side={cfg.side}, stride={cfg.stride}")


Override cfg: resize=384, mode=whole, side=256, stride=192


In [66]:
# 参考：https://github.com/openai/CLIP/blob/main/notebooks/Prompt_Engineering_for_ImageNet.ipynb
PROMPT_TEMPLATES = (
    "a bad photo of a {0}.",
    "a photo of many {0}.",
    "a sculpture of a {0}.",
    "a photo of the hard to see {0}.",
    "a low resolution photo of the {0}.",
    "a rendering of a {0}.",
    "graffiti of a {0}.",
    "a bad photo of the {0}.",
    "a cropped photo of the {0}.",
    "a tattoo of a {0}.",
    "the embroidered {0}.",
    "a photo of a hard to see {0}.",
    "a bright photo of a {0}.",
    "a photo of a clean {0}.",
    "a photo of a dirty {0}.",
    "a dark photo of the {0}.",
    "a drawing of a {0}.",
    "a photo of my {0}.",
    "the plastic {0}.",
    "a photo of the cool {0}.",
    "a close-up photo of a {0}.",
    "a black and white photo of the {0}.",
    "a painting of the {0}.",
    "a painting of a {0}.",
    "a pixelated photo of the {0}.",
    "a sculpture of the {0}.",
    "a bright photo of the {0}.",
    "a cropped photo of a {0}.",
    "a plastic {0}.",
    "a photo of the dirty {0}.",
    "a jpeg corrupted photo of a {0}.",
    "a blurry photo of the {0}.",
    "a photo of the {0}.",
    "a good photo of the {0}.",
    "a rendering of the {0}.",
    "a {0} in a video game.",
    "a photo of one {0}.",
    "a doodle of a {0}.",
    "a close-up photo of the {0}.",
    "a photo of a {0}.",
    "the origami {0}.",
    "the {0} in a video game.",
    "a sketch of a {0}.",
    "a doodle of the {0}.",
    "a origami {0}.",
    "a low resolution photo of a {0}.",
    "the toy {0}.",
    "a rendition of the {0}.",
    "a photo of the clean {0}.",
    "a photo of a large {0}.",
    "a rendition of a {0}.",
    "a photo of a nice {0}.",
    "a photo of a weird {0}.",
    "a blurry photo of a {0}.",
    "a cartoon {0}.",
    "art of a {0}.",
    "a sketch of the {0}.",
    "a embroidered {0}.",
    "a pixelated photo of a {0}.",
    "itap of the {0}.",
    "a jpeg corrupted photo of the {0}.",
    "a good photo of a {0}.",
    "a plushie {0}.",
    "a photo of the nice {0}.",
    "a photo of the small {0}.",
    "a photo of the weird {0}.",
    "the cartoon {0}.",
    "art of the {0}.",
    "a drawing of the {0}.",
    "a photo of the large {0}.",
    "a black and white photo of a {0}.",
    "the plushie {0}.",
    "a dark photo of a {0}.",
    "itap of a {0}.",
    "graffiti of the {0}.",
    "a toy {0}.",
    "itap of my {0}.",
    "a photo of a cool {0}.",
    "a photo of a small {0}.",
    "a tattoo of the {0}.",
)

# 加载模型

In [67]:
# Load the model
import sys
sys.path.append(DINOv3_REPO_DIR)

from dinov3.hub.dinotxt import dinov3_vitl16_dinotxt_tet1280d20h24l
model, tokenizer = dinov3_vitl16_dinotxt_tet1280d20h24l(
    pretrained=True,
    # dinotxt 头 + 文本编码器
    weights=r"E:\Python\pythonProjecttest\demo161\dinov3_vitl16_dinotxt_vision_head_and_text_encoder-a442d8f5.pth",
    # 视觉 backbone，用 ViT-L 预训练权重
    backbone_weights=r"E:\Python\pythonProjecttest\demo161\dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth",
    # 可选：本地 vocab 路径（若已下载 bpe_simple_vocab_16e6.txt.gz）
    # bpe_path_or_url=r"E:\Python\pythonProjecttest\demo161\bpe_simple_vocab_16e6.txt.gz",
)
model.to("cuda", non_blocking=True)
model.eval()
tokenizer = tokenizer.tokenize

OutOfMemoryError: CUDA out of memory. Tried to allocate 12.00 MiB. GPU 0 has a total capacity of 6.00 GiB of which 0 bytes is free. Of the allocated memory 5.19 GiB is allocated by PyTorch, and 154.09 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

# 配置

In [ ]:
@dataclasses.dataclass
class Configuration:
    dataset: str = "cityscapes" # cityscapes, ade20k

    mode: str = "slide"  # whole (whole image), slide (sliding window inference)
    resize: int = 512  # Short side of the input images

    # Only used for mode=slide
    side: int = 384
    stride: int = 192

# Local setup
lovely_tensors.monkey_patch()
warnings.filterwarnings("ignore", message="xFormers")
cfg: Configuration = OmegaConf.to_object(
    OmegaConf.structured(Configuration),
)
print(f"Configuration:\n{OmegaConf.to_yaml(cfg)}")

Configuration:
dataset: cityscapes
mode: slide
resize: 512
side: 384
stride: 192



# 推理

In [ ]:
# 单张自定义图片 + 自定义文本类别推理
# 配置路径和类别名（自行修改）
IMAGE_PATH = r"E:\Python\pythonProjecttest\demo161\data\images\1_tile_r000_c000.jpg"
CLASS_NAMES = ["tree", "crop", "green round plant"]

# 预处理
transform = TVT.Compose(
    [
        ShortSideResize(cfg.resize, TVT.InterpolationMode.BICUBIC),
        TVT.ToTensor(),
        NORMALIZE_IMAGENET,
    ]
)

# 读取单张图
from PIL import Image
img_pil = Image.open(IMAGE_PATH).convert("RGB")
H0, W0 = img_pil.height, img_pil.width
img = transform(img_pil)  # [3, H, W]
print(f"Image loaded: {img.shape}, orig HW=({H0},{W0})")

# 文本提示 -> text_feats（做分批 & 少量模板以节省显存）
PROMPT_TEMPLATES_SMALL = PROMPT_TEMPLATES[:4]  # 再减模板数量
BATCH_PROMPTS = 2

text_feats = []
for class_name in CLASS_NAMES:
    prompts = [template.format(class_name) for template in PROMPT_TEMPLATES_SMALL]
    feats_per_class = []
    for i in range(0, len(prompts), BATCH_PROMPTS):
        batch_prompts = prompts[i : i + BATCH_PROMPTS]
        tokens = tokenizer(batch_prompts)
        # 文本编码转到 CPU 避免占用 GPU 显存
        with torch.inference_mode(), torch.cuda.amp.autocast(enabled=False):
            feats = model.encode_text(tokens.to("cpu"))  # [num_prompts, 2D] on CPU
        feats = feats[:, feats.shape[1] // 2 :]        # 仅用后半部分
        feats = F.normalize(feats, p=2, dim=-1)
        feats_per_class.append(feats)
    feats = torch.cat(feats_per_class, dim=0)
    feats = feats.mean(dim=0)
    feats = F.normalize(feats, p=2, dim=-1)
    text_feats.append(feats.to("cpu"))  # 保持在 CPU，后面转回 GPU
text_feats = torch.stack(text_feats)                  # [num_classes, D] on CPU
print(f"Text features: {text_feats.shape}")

torch.cuda.empty_cache()

# 推理（全图或滑窗）
device = next(model.parameters()).device
img_tensor = img.to(device, non_blocking=True)
text_feats_gpu = text_feats.to(device, non_blocking=True)

with torch.inference_mode(), torch.cuda.amp.autocast():
    if cfg.mode == "whole":
        pred = predict_whole(model, img_tensor, text_feats_gpu)                 # [C, h, w]
    elif cfg.mode == "slide":
        pred = predict_slide(model, img_tensor, text_feats_gpu, cfg.side, cfg.stride)  # [C, H, W]
    else:
        raise ValueError(cfg.mode)

# 上采样回原分辨率并取 argmax
pred = F.interpolate(pred.unsqueeze(0), size=(H0, W0), mode="bilinear", align_corners=False)
pred = pred.squeeze(0).argmax(dim=0).cpu().numpy()  # [H0, W0]，值为类别索引
print("Pred mask shape:", pred.shape, "classes:", CLASS_NAMES)

torch.cuda.empty_cache()

# 可视化叠加与保存
import matplotlib.pyplot as plt
from matplotlib import colors
import imageio

cmap = colors.ListedColormap([
    [0, 0.6, 0],      # tree
    [0.5, 0.3, 0.1],  # soil
    [0, 0, 1],        # irrigation pipe
][: len(CLASS_NAMES)])

plt.figure(figsize=(8, 8))
plt.imshow(img_pil)
plt.imshow(pred, cmap=cmap, alpha=0.4)
plt.axis("off")
plt.show()

# 保存索引掩码
imageio.imwrite("dinotxt_pred_idx.png", pred.astype(np.uint8))
print("Saved: dinotxt_pred_idx.png")

Image loaded: torch.Size([3, 512, 512]), orig HW=(1024,1024)


C:\Users\ZH\AppData\Local\Temp\ipykernel_30072\1756871708.py:33: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.inference_mode(), torch.cuda.amp.autocast():


OutOfMemoryError: CUDA out of memory. Tried to allocate 14.00 MiB. GPU 0 has a total capacity of 6.00 GiB of which 0 bytes is free. Of the allocated memory 5.19 GiB is allocated by PyTorch, and 153.00 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)